# Q2 - FFA "Kinobesucher*innen 2025": Cleaning

**Source:** Filmförderungsanstalt (FFA), *Kinobesucher\*innen 2025* (PDF, used) with permission of the FFA and with attribution. 

This notebook cleans the four files of source Q2, based on the decisions in `q2_explore.ipynb`. 
- Input: `data/raw/ffa/` (read only, never modified)
- OUtput: `data/processed/`

In [35]:
import pandas as pd 
import os 

#Input: raw data (read only)
RAW = "../../data/raw/ffa"

#Output: cleaned data 
PROCESSED = "../../data/processed" 

# Create the output directory if it doesn't exist 
os.makedirs(PROCESSED, exist_ok=True)

#Metadata columns that we drop in all four files (decision from the exploration)
METADATA = ["source_id", "source"]

## 1. File 1: Spending on cinema and home video (2003-2025)

Cleaning steps (decisions from the exploration):
- Drop the metadata columns `source_id` and `source`
- Keep missing values in `est_tvod_million_eur` and `svod_million_eur` as NaN (no replacement with 0)

In [36]:
# Read the raw data file 
spending = pd.read_csv(f"{RAW}/ffa_spending_cinema_homevideo_2003_2025.csv")

#Drop the metadata columns 
spending = spending.drop(columns=METADATA)

#Checks: are the data as we expect? (an error appears if not)
assert spending ["year"].is_unique, "Each year should appear only once"
assert spending["year"].min() == 2003 and spending["year"].max() == 2025
assert len(spending) == 23, "There should be 23 years" 

spending.head()

,year,cinema_million_eur,dvd_bluray_vhs_million_eur,est_tvod_million_eur,svod_million_eur,total_million_eur
0,2003,846,1555,NaN,NaN,2401
1,2004,886,1747,NaN,NaN,2632
2,2005,738,1686,NaN,NaN,2424
3,2006,807,1591,NaN,NaN,2398
4,2007,755,1608,3.0,NaN,2367


## 2. File 2: Visitors by age and gender (2021-2025)

Cleaning Steps (decisions from exploration):
- Drop the metadata columns `source_id` and `source`.
- Rename `tickets_base_million`to `tickets_panel_million`, to make clear that this is the ticket base of the FFA panel survey, not the official ticket count.
- Keep the percentage values as they are (age groups add up to 99-101 because of rounding).


In [37]:
#Read the raw file 
age_gender = pd.read_csv(f"{RAW}/ffa_visitors_age_gender_2021_2025.csv")

#Drop the metadata columns
age_gender = age_gender.drop(columns=METADATA)

# Rename the panel ticket column, so nobody mistakes it for the official ticket count 
age_gender = age_gender.rename(columns={"tickets_base_million": "tickets_panel_million"})

#Checks 
assert age_gender["year"].is_unique, "Each year should appear only once"
assert age_gender["year"].min() == 2021 and age_gender["year"].max() == 2025
assert len(age_gender) == 5, "There should be 5 years"

#Gender should add up to exactly 100
assert (age_gender["male_percent"] + age_gender["female_percent"] == 100).all()

#Age groups should add up to 99-101 (rounding)
age_columns = ["age_10_19_percent", "age_20_29_percent", "age_30_39_percent",
               "age_40_49_percent", "age_50_59_percent", "age_60_plus_percent"]
assert age_gender[age_columns].sum(axis=1).between(99, 101).all()

age_gender.head()


,year,male_percent,female_percent,age_10_19_percent,age_20_29_percent,age_30_39_percent,age_40_49_percent,age_50_59_percent,age_60_plus_percent,tickets_panel_million,avg_age
0,2021,53,47,19,14,20,17,16,13,40.1,38.3
1,2022,50,50,21,14,18,17,15,15,74.0,38.8
2,2023,49,51,21,15,16,17,15,16,91.3,38.9
3,2024,49,51,22,14,16,17,14,17,85.1,39.2
4,2025,48,52,20,13,16,17,16,19,88.6,40.6


## 3. File 3: Cinema reach and visits per person (2015-2025)

Cleaning steps (deciions from the exploration):
- Drop the metadata columns `source_id`and `source`
- Add the column `reach_survey_population` to mark the method change: until 2018 only German citizens were surveyed (`german_citizens`), from 2019 all prople live in Germany (`all_residents`)
- No correction for 2020/2021: the identical values are confirmed in the FFA PDF (p16 and p.58)

In [38]:
# Read the raw file 
reach = pd.read_csv(f"{RAW}/ffa_visitors_reach_2015_2025.csv")

#Drop the metadata columns 
reach = reach.drop(columns=METADATA)

#Mark the method change: first set all rows to "all_residents",
# then overwrite the years until 2018 with "german_citizens"
reach["reach_survey_population"] = "all_residents"
reach.loc[reach["year"] <= 2018, "reach_survey_population"] = "german_citizens"

#Checks 
assert reach["year"].is_unique, "Each year should appear only once"
assert reach["year"].min() == 2015 and reach["year"].max() == 2025
assert len(reach) == 11, "There should be 11 years"

#Percentages should be between 0 and 100 
percent_columns = ["cinema_reach_percent", "home_video_reach_percent",
                   "cinema_or_home_video_reach_percent"]
assert reach[percent_columns].apply(lambda col: col.between(0, 100)).all().all()

reach

,year,cinema_visitors_million,cinema_reach_percent,visits_per_person,home_video_users_million,home_video_reach_percent,cinema_or_home_video_users_million,cinema_or_home_video_reach_percent,reach_survey_population
0,2015,29.9,44,4.5,23.8,35,39.2,58,german_citizens
1,2016,25.0,37,4.7,23.9,35,36.4,54,german_citizens
2,2017,25.0,37,4.7,22.7,34,35.7,53,german_citizens
3,2018,24.6,37,4.1,23.0,34,35.5,53,german_citizens
4,2019,28.2,38,4.0,26.1,35,41.1,55,all_residents
5,2020,16.8,23,2.2,28.1,38,36.8,49,all_residents
6,2021,16.8,23,2.4,28.1,38,36.4,49,all_residents
7,2022,21.7,29,3.4,28.8,39,39.4,53,all_residents
8,2023,23.8,32,3.8,29.6,40,41.2,56,all_residents
9,2024,21.6,29,3.9,30.6,41,40.6,54,all_residents


## 4. Combine files 1-3 into one table 

The three cleaned tables are joined by `year`. File 1 (spending) covers the longest period (2003-2025), so we start with it and add the other two tables.
- Years without data in a file stay empty (NaN), e-g no age data before 2021 and no reach data before 2015. 
- File 4 is not included, because its rows are groups, not years (see section 5).

In [39]:
# Start with spending table (2003-2025) and add the other two tables by year.
# how ="left": keep all years of the spending table 
# validate= "one_to_one": each year may appear only once in both tables (otherwise error)
q2 = spending.merge(reach, on="year", how="left", validate="one_to_one")
q2 = q2.merge(age_gender, on="year", how="left", validate="one_to_one")

#Sort by year 
q2 = q2.sort_values("year").reset_index(drop=True)

#Checks 
assert len(q2) == 23, "There should be 23 years (2003–2025)"
assert q2["year"].is_unique, "Each year should appear only once"
assert len(q2.columns) == 24, "Expected 24 columns (year + 5 + 8 + 10)"

# Save the one cleaned file for source Q2
q2.to_csv(f"{PROCESSED}/q2_ffa_visitors_clean.csv", index=False)

q2



,year,cinema_million_eur,dvd_bluray_vhs_million_eur,est_tvod_million_eur,svod_million_eur,total_million_eur,cinema_visitors_million,cinema_reach_percent,visits_per_person,home_video_users_million,...,male_percent,female_percent,age_10_19_percent,age_20_29_percent,age_30_39_percent,age_40_49_percent,age_50_59_percent,age_60_plus_percent,tickets_panel_million,avg_age
0,2003,846,1555,NaN,NaN,2401,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2004,886,1747,NaN,NaN,2632,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2005,738,1686,NaN,NaN,2424,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2006,807,1591,NaN,NaN,2398,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2007,755,1608,3.0,NaN,2367,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2008,789,1566,8.0,NaN,2363,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2009,974,1638,21.0,NaN,2633,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2010,907,1623,43.0,NaN,2573,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2011,948,1626,78.0,NaN,2652,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2012,1027,1583,102.0,NaN,2712,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. File 4: Cinema visitors and streaming subscribers (2025)

Not included in the cleaned file, because its rows are groups, not years. Used as a single supporting fact for sub-question 2: 

| Group (2025) | Cinema reach | Tickets per person | Avg. ticket price |
|---|---|---|---|
| Population aged 10+ | 31% | 3.8 | 10.39 EUR |
| SVOD subscribers | 36% | 4.8 | 10.48 EUR |